## Проблема

Имеются наблюдения за бизонами: особенности поведения, физические характеристики, показатели активности. Однако определить точный возраст бизона напрямую практически сложно — животные улетают при виде человека.

Вес бизона — один из важных признаков, по которому можно судить о его возрасте.

## Задача ML

**Задача:** предсказать возраст летающего бизона по его физическим характеристикам и показателям активности.

**Объект:** летающий бизон (одна запись в датасете).

**Образ выходного результата модели:** число — возраст бизона (возраст в годах, непрерывная величина).

**Тип задачи:** регрессия.

# Описание исходных данных

## Семантика и ожидаемые типы признаков

| Признак | Семантика | Ожидаемый тип | Целевой |
|---|---|---|---|
| Type | Тип бизона (исторические данные / дикие / прирученные) | categorical (string) | Нет |
| Length | Длина бизона | float | Нет |
| Diameter | Диаметр бизона | float | Нет |
| Height | Высота бизона | float | Нет |
| Whole_weight | Общий вес бизона | float | Нет |
| Age | Возраст бизона (в годах) | float | **Да** |

## Формулы предметной области

Формулы предметной области для предсказания возраста в датасете не предоставлены. МЕМО даёт только подсказку о возможной двухэтапной схеме: сначала предсказать вес по признакам, затем возраст по весу. Конкретные формулы связи веса и возраста не указаны — они будут определяться моделью ML на этапе обучения.

In [ ]:
import pandas as pd
import numpy as np
import typing as t
import warnings
import missingno as msgn
warnings.filterwarnings('ignore')

seed = 42
np.random.seed(seed)

# 1. Предварительная обработка данных

## 1.1. Чтение и загрузка данных
Загружаем датасет с характеристиками летающих бизонов. Проверяем корректность загрузки: количество строк и столбцов, типы данных, наличие пропусков.

In [ ]:
import os

home_dir = os.path.expanduser('~')
data_path = os.path.join(home_dir, 'OneDrive', 'Desktop', 'Studies_2026', 'ProgPractice', 'materials', 'datasets', '05_avatar.csv')

df = pd.read_csv(data_path, sep=',')

print(f"Размер датасета: {df.shape}")
print(f"\nПервые 5 строк:")
df.head()

### Удаление строк с пропусками в таргете
Поскольку Age — целевой признак, строки с пропусками в нём нужно удалить. Удаляем до разделения на train/test, чтобы избежать смещения распределений.

In [ ]:
df = df.dropna(subset=['Age'])

print(f"Размер после удаления пропусков в Age: {df.shape}")

### Удаление строк с высокой концентрацией пропусков
Согласно лекции, строки с высокой концентрацией пропусков (>75-го перцентиля) могут стать аномальными после заполнения и подлежат удалению. Вычисляем долю пропусков по строкам и удаляем такие строки до разделения на train/test.

In [ ]:
# Вычисляем долю пропусков по строкам
row_missing_ratio = df.isnull().mean(axis=1)
print(f"75-й перцентиль доли пропусков по строкам: {row_missing_ratio.quantile(0.75):.4f}")

# Удаляем строки с долей пропусков >75-го перцентиля
threshold = row_missing_ratio.quantile(0.75)
df = df[row_missing_ratio <= threshold]

print(f"Размер после удаления строк с высокой концентрацией пропусков: {df.shape}")

## 1.2. Разделение выборки на обучение и тест

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed)

print(f"Train размер: {train_df.shape}")
print(f"Test размер: {test_df.shape}")

## 1.3. Первичный анализ данных

Проводим анализ только на train-выборке, чтобы избежать утечки данных. Проверяем:
- Типы данных 
- Количество уникальных значений
- Наличие пропусков в каждом признаке
- Основные статистики

In [ ]:
train_df.info()

In [ ]:
train_df.nunique()

In [ ]:
train_df.describe()

In [ ]:
print("Уникальные значения Type:")
print(train_df['Type'].value_counts())

## 1.4. Обработка вещественных признаков

### Анализ пропусков
Сначала анализируем пропуски в вещественных признаках. Визуализируем матрицу пропусков и определяем долю пропусков по признакам и по строкам.

In [ ]:
# Выделяем только вещественные признаки (исключая таргет Age)
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'Age']

# Визуализация пропусков
msgn.matrix(train_df[numeric_cols])

### Доля пропусков по признакам и по строкам
Выводим долю пропусков для каждого вещественного признака, чтобы решить, какие признаки можно удалить (если >50-70%). Также проверяем долю пропусков по строкам после удаления.

In [ ]:
# Доля пропусков по признакам
missing_ratio = train_df[numeric_cols].isnull().mean()
print("Доля пропусков по признакам:")
print(missing_ratio)

# Доля пропусков по строкам
row_missing_ratio_train = train_df.isnull().mean(axis=1)
print(f"\nДоля пропусков по строкам (макс): {row_missing_ratio_train.max():.4f}")
print(f"Доля пропусков по строкам (75-й перцентиль): {row_missing_ratio_train.quantile(0.75):.4f}")

### Удаление признаков с высокой долей пропусков
Согласно лекции, признаки с >50-70% пропусков можно считать неинформативными и удалить. Выводим, какие признаки попадают под это правило.

In [ ]:
# Признаки с >50% пропусков
high_missing = missing_ratio[missing_ratio > 0.5].index.tolist()
print(f"Признаки с >50% пропусков: {high_missing}")

# В данном случае таких признаков нет, но если бы были, мы бы их удалили
# numeric_cols = [col for col in numeric_cols if col not in high_missing]

### Заполнение пропусков в вещественных признаках
Заполняем пропуски статистическими методами (mean/median). Выбор значения обосновываем анализом распределений и KL-дивергенцией.

In [ ]:
from scipy.stats import entropy

def calc_kl_divergence(original: np.ndarray, filled: np.ndarray, bins: int = 30) -> float:
    """Вычисление KL-дивергенции между двумя распределениями.
    
    Чем ближе к 0, тем больше распределения похожи.
    """
    # Строим гистограммы с одинаковыми бинами
    hist_orig, _ = np.histogram(original, bins=bins, density=True)
    hist_filled, _ = np.histogram(filled, bins=bins, density=True)
    
    # Добавляем малую константу, чтобы избежать деления на 0
    epsilon = 1e-10
    hist_orig = hist_orig + epsilon
    hist_filled = hist_filled + epsilon
    
    # Нормализуем
    hist_orig = hist_orig / hist_orig.sum()
    hist_filled = hist_filled / hist_filled.sum()
    
    # KL-дивергенция
    kl = entropy(hist_filled, hist_orig)
    return kl

In [ ]:
# Для каждого признака с пропусками тестируем заполнение mean и median
fillna_rule = {}

for col in numeric_cols:
    if train_df[col].isnull().sum() > 0:
        original = train_df[col].dropna().values
        
        # Заполнение mean
        filled_mean = train_df[col].fillna(train_df[col].mean()).values
        kl_mean = calc_kl_divergence(original, filled_mean)
        
        # Заполнение median
        filled_median = train_df[col].fillna(train_df[col].median()).values
        kl_median = calc_kl_divergence(original, filled_median)
        
        print(f"{col}: KL(mean)={kl_mean:.4f}, KL(median)={kl_median:.4f}")
        
        # Выбираем значение с меньшей KL-дивергенцией
        if kl_mean < kl_median:
            fillna_rule[col] = train_df[col].mean()
            print(f"  -> выбрано mean")
        else:
            fillna_rule[col] = train_df[col].median()
            print(f"  -> выбрано median")

print(f"\nПравило заполнения: {fillna_rule}")

In [ ]:
# Применяем правило заполнения к train и test
for col, value in fillna_rule.items():
    train_df[col] = train_df[col].fillna(value)
    test_df[col] = test_df[col].fillna(value)

print("Пропуски заполнены.")
print(f"Пропуски в train после заполнения: {train_df[numeric_cols].isnull().sum().sum()}")
print(f"Пропуски в test после заполнения: {test_df[numeric_cols].isnull().sum().sum()}")

## 1.5. Кодирование категориальных признаков

### Проверка категорий в train
Проверяем, что все категории из оригинального датасета присутствуют в train. Если какие-то категории отсутствуют в train, это нужно учесть в кодировщике (handle_unknown='ignore').

In [ ]:
# Выделяем категориальные признаки
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(f"Категориальные признаки: {cat_cols}")

# Проверяем, что все категории есть в train
for col in cat_cols:
    df_cats = set(df[col].dropna().unique())
    train_cats = set(train_df[col].dropna().unique())
    missing_cats = df_cats - train_cats
    if missing_cats:
        print(f"{col}: категории, отсутствующие в train: {missing_cats}")
    else:
        print(f"{col}: все категории присутствуют в train")

### One-Hot кодирование
Признак Type — равнозначные категории (исторические данные / дикие / прирученные), используем OneHotEncoder. Параметр handle_unknown='ignore' позволяет корректно обрабатывать новые категории в test (если они вдруг появятся).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Инициализируем кодировщик
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

# Fit на train, transform на train и test
train_encoded = encoder.fit_transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

# Получаем имена новых признаков
encoded_feature_names = encoder.get_feature_names_out(cat_cols)
print(f"Новые признаки после кодирования: {encoded_feature_names.tolist()}")

# Создаём DataFrame с закодированными признаками
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_feature_names, index=train_df.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_feature_names, index=test_df.index)

# Удаляем исходные категориальные признаки и добавляем закодированные
train_df = train_df.drop(columns=cat_cols).join(train_encoded_df)
test_df = test_df.drop(columns=cat_cols).join(test_encoded_df)

print(f"\nРазмер train после кодирования: {train_df.shape}")
print(f"Размер test после кодирования: {test_df.shape}")

## 1.6. Сборка предобработанного датасета
Проверяем итоговый размер датасета и отсутствие пропусков.

In [ ]:
print(f"Итоговый размер train: {train_df.shape}")
print(f"Итоговый размер test: {test_df.shape}")
print(f"\nПропуски в train: {train_df.isnull().sum().sum()}")
print(f"Пропуски в test: {test_df.isnull().sum().sum()}")
print(f"\nТипы данных train:")
print(train_df.dtypes)

## Production
Блок чистого воспроизводимого кода для обработки данных. Все шаги предобработки собраны вместе без markdown-обоснований.

In [ ]:
# ==================== PRODUCTION BLOCK ====================
# Чистый воспроизводимый код для предобработки данных

# Загрузка данных
home_dir = os.path.expanduser('~')
data_path = os.path.join(home_dir, 'OneDrive', 'Desktop', 'Studies_2026', 'ProgPractice', 'materials', 'datasets', '05_avatar.csv')
df = pd.read_csv(data_path, sep=',')

# Удаление строк с пропусками в таргете
df = df.dropna(subset=['Age'])

# Удаление строк с высокой концентрацией пропусков (>75-го перцентиля)
row_missing_ratio = df.isnull().mean(axis=1)
threshold = row_missing_ratio.quantile(0.75)
df = df[row_missing_ratio <= threshold]

# Разделение на train/test
seed = 42
train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed)

# Выделение признаков
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'Age']  # исключаем таргет
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

# Заполнение пропусков (динамическое вычисление по KL-дивергенции)
fillna_rule = {}
for col in numeric_cols:
    if train_df[col].isnull().sum() > 0:
        original = train_df[col].dropna().values
        filled_mean = train_df[col].fillna(train_df[col].mean()).values
        filled_median = train_df[col].fillna(train_df[col].median()).values
        
        # KL-дивергенция
        hist_orig, _ = np.histogram(original, bins=30, density=True)
        hist_mean, _ = np.histogram(filled_mean, bins=30, density=True)
        hist_median, _ = np.histogram(filled_median, bins=30, density=True)
        
        epsilon = 1e-10
        hist_orig = (hist_orig + epsilon) / (hist_orig + epsilon).sum()
        hist_mean = (hist_mean + epsilon) / (hist_mean + epsilon).sum()
        hist_median = (hist_median + epsilon) / (hist_median + epsilon).sum()
        
        kl_mean = entropy(hist_mean, hist_orig)
        kl_median = entropy(hist_median, hist_orig)
        
        if kl_mean < kl_median:
            fillna_rule[col] = train_df[col].mean()
        else:
            fillna_rule[col] = train_df[col].median()

for col, value in fillna_rule.items():
    train_df[col] = train_df[col].fillna(value)
    test_df[col] = test_df[col].fillna(value)

# One-Hot кодирование
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
train_encoded = encoder.fit_transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

encoded_feature_names = encoder.get_feature_names_out(cat_cols)
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_feature_names, index=train_df.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_feature_names, index=test_df.index)

train_df = train_df.drop(columns=cat_cols).join(train_encoded_df)
test_df = test_df.drop(columns=cat_cols).join(test_encoded_df)

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"Пропуски train: {train_df.isnull().sum().sum()}, Пропуски test: {test_df.isnull().sum().sum()}")
# ==================== END PRODUCTION BLOCK ====================